# Extracción de keypoints con YOLO11-pose
**Proyecto:** detección y clasificación de golpes de boxeo (YOLO11-pose + LSTM)

Este notebook recorre tus videos, saca los keypoints del cuerpo en cada frame con YOLO11-pose y guarda:

- `salida/keypoints/<clase>/<video>.npz` → keypoints de cada frame (esto alimenta al LSTM más adelante)
- `salida/overlays/<clase>/<video>.mp4` → el mismo video con el esqueleto dibujado (para revisar)
- `salida/manifest.csv` → tabla resumen por video (frames, confianza, estado)
- `salida/balance_clases.png` y `salida/muestras_esqueleto.png` → gráficos de evidencia

**Antes de correr (una sola vez):**
1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar**.
2. En tu Google Drive debe existir la carpeta `boxeo_proyecto/videos/` con una subcarpeta por clase (`jab`, `cross`, `hook`, `uppercut`, `guardia`) y, para hoy, una carpeta `prueba` con un video corto.
3. Ejecuta las celdas en orden, de arriba hacia abajo (botón ▶ de cada celda, o Shift+Enter).

In [ ]:
# CELDA 1: instalar la librería (tarda ~1 minuto la primera vez)
!pip -q install ultralytics

In [ ]:
# CELDA 2: importar librerías y conectar Google Drive
import subprocess, shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm
from ultralytics import YOLO
from IPython.display import display

from google.colab import drive
drive.mount('/content/drive')

# ---- Ruta base: debe coincidir con la carpeta que creaste en Drive ----
BASE = Path('/content/drive/MyDrive/boxeo_proyecto')

print('GPU disponible:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('AVISO: no hay GPU. Ve a Entorno de ejecución > Cambiar tipo de entorno > T4 GPU.')

In [ ]:
# CELDA 3: parámetros (aquí puedes ajustar cosas sin tocar el resto)
MODELO = 'yolo11m-pose.pt'            # modelo de pose (se descarga solo la primera vez)
EXTENSIONES = {'.mp4', '.mov', '.m4v', '.avi'}
REPROCESAR = False                    # False = salta videos ya procesados; True = rehace todo
UMBRAL_SIN_DETECCION = 0.20           # marcar REVISAR si >20% de los frames no detectó a nadie
UMBRAL_CONF = 0.50                    # marcar REVISAR si la confianza media del torso es < 0.50

# Puntos del torso y brazos (formato COCO): nariz, hombros, codos, muñecas, caderas
PUNTOS_TORSO = [0, 5, 6, 7, 8, 9, 10, 11, 12]

VIDEOS = BASE / 'videos'
SALIDA = BASE / 'salida'
KP_DIR = SALIDA / 'keypoints'
OV_DIR = SALIDA / 'overlays'

if not VIDEOS.exists():
    raise FileNotFoundError(f'No encuentro la carpeta {VIDEOS}. Revisa que la creaste en Drive con ese nombre exacto.')

KP_DIR.mkdir(parents=True, exist_ok=True)
OV_DIR.mkdir(parents=True, exist_ok=True)

clases = sorted(p.name for p in VIDEOS.iterdir() if p.is_dir())
print('Clases encontradas:', clases)

In [ ]:
# CELDA 4: cargar el modelo
model = YOLO(MODELO)
print('Modelo cargado:', MODELO)

In [ ]:
# CELDA 5: funciones de procesamiento (solo se definen, no hacen nada todavía)

def a_h264(origen: Path, destino: Path):
    """Re-codifica el video a H.264 para que se pueda ver en Drive/QuickTime."""
    try:
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(origen),
                        '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', str(destino)], check=True)
        origen.unlink()
    except Exception as e:
        shutil.move(str(origen), str(destino))
        print('  (aviso) no pude convertir a H.264, dejé el video tal cual:', e)


def procesar_video(ruta_video: Path, ruta_overlay: Path):
    """Corre YOLO-pose frame a frame. Devuelve keypoints, confianzas, detectado, fps y un frame de muestra."""
    cap = cv2.VideoCapture(str(ruta_video))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

    tmp = ruta_overlay.with_name(ruta_overlay.stem + '_tmp.mp4')
    writer = None
    kpts, confs, detectado = [], [], []
    muestra = None

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        res = model(frame, verbose=False)[0]
        hay_persona = res.keypoints is not None and len(res.boxes) > 0

        if hay_persona:
            # si hay varias personas, nos quedamos con la de la caja más grande
            areas = res.boxes.xywh[:, 2] * res.boxes.xywh[:, 3]
            i = int(areas.argmax())
            kpts.append(res.keypoints.xy[i].cpu().numpy().astype(np.float32))
            if res.keypoints.conf is not None:
                confs.append(res.keypoints.conf[i].cpu().numpy().astype(np.float32))
            else:
                confs.append(np.ones(17, dtype=np.float32))
            detectado.append(True)
        else:
            kpts.append(np.zeros((17, 2), dtype=np.float32))
            confs.append(np.zeros(17, dtype=np.float32))
            detectado.append(False)

        anotado = res.plot()  # frame con el esqueleto dibujado
        if writer is None:
            h, w = anotado.shape[:2]
            writer = cv2.VideoWriter(str(tmp), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
        writer.write(anotado)

        if len(kpts) == 30:        # guardamos el frame 30 como muestra para el gráfico
            muestra = anotado.copy()
        last = anotado

    cap.release()
    if writer is not None:
        writer.release()
        a_h264(tmp, ruta_overlay)
        if muestra is None:
            muestra = last

    kpts = np.array(kpts) if kpts else np.zeros((0, 17, 2), dtype=np.float32)
    confs = np.array(confs) if confs else np.zeros((0, 17), dtype=np.float32)
    detectado = np.array(detectado, dtype=bool)
    return kpts, confs, detectado, float(fps), muestra


def estadisticas(clase, nombre, kpts, confs, det, fps):
    n = len(kpts)
    pct_sin = float(1 - det.mean()) if n else 1.0
    conf_media = float(confs[det][:, PUNTOS_TORSO].mean()) if det.any() else 0.0
    estado = 'REVISAR' if (n == 0 or pct_sin > UMBRAL_SIN_DETECCION or conf_media < UMBRAL_CONF) else 'OK'
    return {
        'clase': clase, 'video': nombre, 'frames': n, 'fps': round(fps, 1),
        'duracion_s': round(n / fps, 2) if fps else 0,
        'pct_sin_deteccion': round(pct_sin * 100, 1),
        'conf_media_torso': round(conf_media, 3),
        'estado': estado,
    }

print('Funciones listas.')

In [ ]:
# CELDA 6: procesar TODOS los videos (esta es la celda que tarda)
videos = []
for clase in clases:
    for v in sorted((VIDEOS / clase).iterdir()):
        if v.suffix.lower() in EXTENSIONES and not v.name.startswith('.'):
            videos.append((clase, v))

if not videos:
    raise RuntimeError(f'No encontré videos en {VIDEOS}. Revisa que estén dentro de las subcarpetas (jab, cross, ...) y que la extensión sea .mp4 o .mov.')

print(f'Videos a procesar: {len(videos)}')

filas = []
muestras = {}

for clase, ruta in tqdm(videos):
    nombre = ruta.stem
    (KP_DIR / clase).mkdir(parents=True, exist_ok=True)
    (OV_DIR / clase).mkdir(parents=True, exist_ok=True)
    npz_path = KP_DIR / clase / f'{nombre}.npz'
    ov_path = OV_DIR / clase / f'{nombre}.mp4'

    if npz_path.exists() and not REPROCESAR:
        d = np.load(npz_path)
        kpts, confs, det, fps = d['kpts'], d['conf'], d['detectado'], float(d['fps'])
    else:
        kpts, confs, det, fps, muestra = procesar_video(ruta, ov_path)
        np.savez_compressed(npz_path, kpts=kpts, conf=confs, detectado=det, fps=fps)
        if muestra is not None and clase not in muestras:
            muestras[clase] = (nombre, muestra)

    filas.append(estadisticas(clase, nombre, kpts, confs, det, fps))

df = pd.DataFrame(filas)
df.to_csv(SALIDA / 'manifest.csv', index=False, encoding='utf-8')
print('Listo. Tabla guardada en', SALIDA / 'manifest.csv')
display(df)

In [ ]:
# CELDA 7: gráficos de evidencia (balance de clases y muestras con esqueleto)
resumen = df.groupby('clase').agg(videos=('video', 'count'), frames=('frames', 'sum')).reset_index()

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].bar(resumen['clase'], resumen['videos'])
ax[0].set_title('Videos por clase')
ax[1].bar(resumen['clase'], resumen['frames'])
ax[1].set_title('Frames por clase')
plt.tight_layout()
plt.savefig(SALIDA / 'balance_clases.png', dpi=150)
plt.show()

if muestras:
    fig, axs = plt.subplots(1, len(muestras), figsize=(4 * len(muestras), 4), squeeze=False)
    for ax, (clase, (nombre, img)) in zip(axs[0], muestras.items()):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f'{clase}\n{nombre}', fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(SALIDA / 'muestras_esqueleto.png', dpi=150)
    plt.show()
else:
    print('No hay muestras nuevas (los videos ya estaban procesados). Pon REPROCESAR = True en la celda 3 si las quieres.')

print('Revisa en Drive la carpeta:', SALIDA)

## Qué revisar al terminar (checklist de hoy)

1. En la tabla, la columna **estado** debe decir `OK` para tu video de prueba. Si dice `REVISAR`, mira `pct_sin_deteccion` y `conf_media_torso`.
2. En la imagen `muestras_esqueleto.png`, el esqueleto debe calzar sobre tu cuerpo (hombros, codos y muñecas bien ubicados).
3. Abre el video de `salida/overlays/prueba/` y confirma que el esqueleto te sigue durante el movimiento.

Si algo no cuadra, copia el mensaje de error o describe lo que ves, y lo corregimos antes de grabar mañana.